# 00 ,  Configuración del entorno computacional

**TFM Inga-Español.** Este notebook valida el entorno local (Apple M4 Max, 128 GB, macOS) para las fases siguientes del proyecto. Ejecución única al arrancar el desarrollo.

## Versiones y aceleración

In [ ]:
import sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('MPS disponible:', torch.backends.mps.is_available())
print('MPS construido:', torch.backends.mps.is_built())

## Librerías instaladas

Las librerías se instalan desde `requirements.txt` o mediante `uv`/`pip`. Este notebook solo verifica disponibilidad.

In [ ]:
import importlib
libs = ['transformers', 'peft', 'accelerate', 'sentence_transformers',
        'sacrebleu', 'datasets', 'anthropic', 'sentencepiece',
        'matplotlib', 'pandas', 'numpy', 'tqdm']
for lib in libs:
    try:
        m = importlib.import_module(lib)
        print(f'  OK  {lib:25s} {getattr(m, "__version__", "?")}')
    except ImportError as e:
        print(f'  MISS {lib:25s} NO INSTALADA ({e})')

## Descarga y prueba del modelo base pequeño

Se descarga `facebook/nllb-200-distilled-600M` como modelo de trabajo para iteración rápida. Se valida una inferencia zero-shot Español->Quechua Ayacucho, par lingüísticamente cercano al Inga que el modelo ya incluye en sus lenguas soportadas.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
model_name = 'facebook/nllb-200-distilled-600M'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model.to(device)
print(f'Modelo cargado en {device}.')

In [ ]:
# Prueba: Espanol -> Quechua Ayacucho (quy_Latn)
src_text = 'La lengua Inga es hablada en el Putumayo.'
tokenizer.src_lang = 'spa_Latn'
inputs = tokenizer(src_text, return_tensors='pt').to(device)
translated_tokens = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids('quy_Latn'),
    max_length=128
)
result = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
print('ES :', src_text)
print('QUY:', result)

## Resumen

Si todas las celdas anteriores se ejecutan sin error, el entorno está listo para las Fases siguientes del TFM. El modelo NLLB-200-distilled-600M queda cacheado en `~/.cache/huggingface/hub` para notebooks posteriores.